# Lecture 3/4 - Geocentric models and wiggly orbits

McElreath's lectures for the whole book are available here: https://github.com/rmcelreath/stat_rethinking_2022

An R/Stan repo of code is available here: https://vincentarelbundock.github.io/rethinking2/

An excellent port to Python/PyMC Code is available here: https://github.com/dustinstansbury/statistical-rethinking-2023

You are encouraged to work through both of these versions to re-enforce what we're doing in class.

In [ ]:
# Load R packages
library(cmdstanr)    # R interface to Stan
library(posterior)   # Working with posterior draws
library(bayesplot)   # Plotting posterior draws
library(splines)     # B-spline basis functions

# Save the current figure to file (uncomment the savefig() calls below to use)
savefig <- function(file, width = 7, height = 5){
    dev.copy(jpeg, file, width = width, height = height, units = "in", res = 300)
    invisible(dev.off())
}

## !Kung Kids

Let's import the Nancy Howell's data from the Kalahari people and take a look:

In [ ]:
# Import data
kdata <- read.csv('howell.csv')
# Display top 5 rows
head(kdata, 5)

In [ ]:
# Table of descriptive statistics
summary(kdata)

Let's take a look at the distribution of the data

In [ ]:
# Plot the distribution (kernel density) of all heights
plot(density(kdata$height), main = '!Kung heights from Nancy Howell', xlab = 'Height (cm)')
# savefig('fullkung.jpg')

Long tails here, what's going on?

In [ ]:
# Colours for kids
xcol <- c('grey', 'dodgerblue')
# Dummy for kids (1 = kid, 2 = adult, for indexing the colours)
Ik <- (kdata$age > 17) + 1
plot(kdata$weight, kdata$height, col = xcol[Ik], pch = 16, xlab = 'Weight (kg)', ylab = 'Height (cm)')
# savefig('heightweight.jpg')

Ah, those pesky kids. So our glorious linearity among the adults is now gone. So how should we model this relationship?


# Curves from 'linear models'

### Link functions
    - common
    - ok

### Polyomial regression
    - common
    - bad

### Splines
    - very flexible
    - highly geocentric



## Link functions

Link functions are present in every statistical model as they translate the model scale - which can be log-odds for example - onto the observation scale. A log-link or logit-link are common choices. Let's see what a log link does to our plot:

In [ ]:
plot(log(kdata$weight), kdata$height, col = xcol[Ik], pch = 16, xlab = 'log(Weight) (kg)', ylab = 'Height (cm)')
# savefig('logheightweight.jpg')

Near linearity - very helpful.

## Polynomial Regression

If you care not about any sort of process but are confident in the relative shape of what you want, a polynomial can **sometimes** be worth a look. 

$1^{st}$ order (line): $\mu_i = \beta_0+\beta_1 x_i$

$2^{nd}$ order (parabola): $\mu_i = \beta_0+\beta_1 x_i+\beta_2 x^{2}_i$

$3^{rd}$ order (cubic): $\mu_i = \beta_0+\beta_1 x_i+\beta_2 x^{2}_i+\beta_3 x^{3}_i$

$n^{th}$ order (insanity): $\mu_i = \beta_0+\beta_1 x_i+\beta_2 x^{2}_i+...+\beta_n x^{n}_i$

And it goes on...with increasingly uninterpretable results. Even the parabolic parameters are not individually interpretable, so be careful!!


## Polynomial !Kungs

So let's try fitting a paraboloa and see how well it does. As a first step we're going to standardize the data, meaning we'll subtract the mean (as always) and also divide by the standard deviation*. Why? Because it makes the intercept interpretable (as with zero-centreing) and because it places things on a small-ish scale near zero, where likelhoods don't explode. 

\*Note I typically standardize by 2sd because it makes 0-1 (dummy) variables comparable with standardized variables, allowing us to look at **relative** effect sizes

In [ ]:
# Standardize weights
kweights <- (kdata$weight - mean(kdata$weight))/sd(kdata$weight)
hist(kweights, main = '!Kung heights from Nancy Howell', xlab = 'Height (z-score)')
# savefig('zkung.jpg')

So for our model:

$$
\large{
\begin{align*}
z_i &= \frac{x_i-\bar{x}}{SD(x)} \\
h_i &\sim N(\mu_i,\sigma) \\
\mu_i &= \beta_0 + \beta_1z_i + \beta_2z^{2}_i \\
\beta_0 &\sim N(178,20) \\
\beta_1 &\sim logN(0,1) \\
\beta_2 &\sim N(0,1) \\
\sigma &\sim U(0, 50)
\end{align*}}
$$

We've added the quadratic parameter $\beta_2$, for which we have specified a $N(0,1)$ prior. Why? Weelll based on the recommendations of our modern-day Bayesian Yoda [Andrew Gelman](http://www.stat.columbia.edu/~gelman/), who very often standardizes covariates and uses $N(0,1)$ priors in his own work. He's done the legwork, so we take it and move on.

Coding this into a Stan model, we get:

In [ ]:
# Model in Stan
kung_code <- "
data {
  int<lower=0> N;
  vector[N] z;         // standardized weights
  vector[N] height;
}
parameters {
  real Average_height;
  real<lower=0> Weight;
  real Weight2;
  real<lower=0, upper=50> Obs_sd;
}
model {
  // Priors
  Average_height ~ normal(178, 20);
  Weight ~ lognormal(0, 1);
  Weight2 ~ normal(0, 1);
  Obs_sd ~ uniform(0, 50);

  // Linear model
  vector[N] mu = Average_height + Weight*z + Weight2*square(z);

  // Likelihood
  height ~ normal(mu, Obs_sd);
}
"
kung <- cmdstan_model(write_stan_file(kung_code))
kdat <- list(N = nrow(kdata), z = kweights, height = kdata$height)

In [ ]:
# Sampling
trace_p2 <- kung$sample(data = kdat, chains = 4, parallel_chains = 4, refresh = 0, show_exceptions = FALSE)

In [ ]:
b0 <- as.vector(trace_p2$draws("Average_height"))
b1 <- as.vector(trace_p2$draws("Weight"))
b2 <- as.vector(trace_p2$draws("Weight2"))
sig <- as.vector(trace_p2$draws("Obs_sd"))

In [ ]:
plot(kweights, kdata$height, col = "dodgerblue", xlab = 'Weight (z-score)', ylab = 'Height (cm)')
xnew <- seq(min(kweights), max(kweights), length.out = 100)
y_ <- mean(b0) + mean(b1)*xnew + mean(b2)*xnew^2
y_uu <- y_ + mean(sig)*2
y_ul <- y_ - mean(sig)*2
lines(xnew, y_, lwd = 2)
lines(xnew, y_uu, lty = 3, lwd = 2)
lines(xnew, y_ul, lty = 3, lwd = 2)
# savefig('2ndorder.jpg')

Another way of plotting the intervals is to to make random draws from the posterior distribution, which contains a multitude of wiggly lines

In [ ]:
plot(kweights, kdata$height, col = "dodgerblue", xlab = 'Weight (z-score)', ylab = 'Height (cm)')
xnew <- seq(min(kweights), max(kweights), length.out = 100)
for (i in 1:100){
    j <- sample(length(b0), 1)   # a random draw from the joint posterior
    y_ <- b0[j] + b1[j]*xnew + b2[j]*xnew^2
    lines(xnew, y_, col = adjustcolor("black", 0.1))
}

A key part of polynomials is that they often do a horrific job outside of the data, flopping around everywhere. This is clear if we look at slightly heavier people

In [ ]:
xnew <- seq(min(kweights), max(kweights) + 0.5, length.out = 100)
plot(kweights, kdata$height, col = "dodgerblue", xlim = range(xnew), xlab = 'Weight (z-score)', ylab = 'Height (cm)')
for (i in 1:100){
    j <- sample(length(b0), 1)
    y_ <- b0[j] + b1[j]*xnew + b2[j]*xnew^2
    lines(xnew, y_, col = adjustcolor("black", 0.1))
}
# savefig('fat2ndorder.jpg')

(shrinking large people...)

What about 3rd order?

In [ ]:
# Model in Stan
kung3_code <- "
data {
  int<lower=0> N;
  vector[N] z;         // standardized weights
  vector[N] height;
}
parameters {
  real Average_height;
  real<lower=0> Weight;
  real Weight2;
  real Weight3;
  real<lower=0, upper=50> Obs_sd;
}
model {
  // Priors
  Average_height ~ normal(178, 20);
  Weight ~ lognormal(0, 1);
  Weight2 ~ normal(0, 1);
  Weight3 ~ normal(0, 1);
  Obs_sd ~ uniform(0, 50);

  // Linear model
  vector[N] mu = Average_height + Weight*z + Weight2*square(z) + Weight3*pow(z, 3);

  // Likelihood
  height ~ normal(mu, Obs_sd);
}
"
kung3 <- cmdstan_model(write_stan_file(kung3_code))

In [ ]:
# Sampling
trace_p3 <- kung3$sample(data = kdat, chains = 4, parallel_chains = 4, refresh = 0, show_exceptions = FALSE)

In [ ]:
b0 <- as.vector(trace_p3$draws("Average_height"))
b1 <- as.vector(trace_p3$draws("Weight"))
b2 <- as.vector(trace_p3$draws("Weight2"))
b3 <- as.vector(trace_p3$draws("Weight3"))
sig <- as.vector(trace_p3$draws("Obs_sd"))

In [ ]:
plot(kweights, kdata$height, col = "dodgerblue", xlim = range(xnew), xlab = 'Weight (z-score)', ylab = 'Height (cm)')
y_ <- mean(b0) + mean(b1)*xnew + mean(b2)*xnew^2 + mean(b3)*xnew^3
y_uu <- y_ + mean(sig)*2
y_ul <- y_ - mean(sig)*2
lines(xnew, y_, lwd = 2)
lines(xnew, y_uu, lty = 3, lwd = 2)
lines(xnew, y_ul, lty = 3, lwd = 2)
# savefig('3rdorder.jpg')

# Splines

![](splines.png)


 - Basis-Splines: wiggly function build from many local less wiggly functions
 - Basis function: a local function
 - Better than polynomials, but equally geocentric
 - Bayesian B-splines called *P-splines*
 
So what does these B-splines look like? Well they're just linear models that have synthetic variables (B's):

$$
\mu_i = \beta_0 + w_1 B_{i,1}+ w_2 B_{i,2}+...++ w_n B_{i,n}
$$

w - are weights that are just like slopes, while the basis functions turn on these weights for specific regions of *x*. 

To make this clear, let's have a look at the Japanese cherry blossom data:




In [ ]:
cdata <- read.csv('https://raw.githubusercontent.com/rmcelreath/rethinking/master/data/cherry_blossoms.csv', sep = ";")
tmp <- sum(is.na(cdata$temp)); tmp2 <- length(cdata$temp)
# Drop rows where temp is NA
cdata <- cdata[!is.na(cdata$temp), ]
head(cdata)

In [ ]:
c(tmp, tmp2); dim(cdata)

In [ ]:
options(repr.plot.width = 12, repr.plot.height = 5)
plot(cdata$year, cdata$temp, col = "dodgerblue", xlab = 'Year', ylab = 'March temp')
abline(h = max(cdata$temp[cdata$year < 1800]), col = 'red')
# savefig('blossoms.jpg', width = 12)

In [ ]:
# Table of descriptive statistics
summary(cdata)

So if we're going to put splines through this, how do they work? Well the algorithm needs to:

 - choose knots (places where the spline is anchored)
 - choose degree of basis functions (how wiggly)
 - find posterior distribution of the weights
 
So to start, let's pick some arbitrary knots across equal quantiles across the data:

In [ ]:
# Number of knots
nk <- 5
# Knot locations
naughts <- quantile(cdata$year, probs = seq(0, 1, length.out = nk))
naughts

So how do we chose the degree of basis functions?

Well, can start with basis fucntions which are degree 1, which make them a linear combination at 2 points:

![](basis1.jpg)

To get our basis functions, we can use the `bs()` function in the `splines` package (as in the Python version with `patsy`, we add a column of 1's for the intercept at the front):

In [ ]:
# Construct the Basis function matrix (interior knots only; the end knots are the data range)
B <- cbind(Intercept = 1, bs(cdata$year, knots = naughts[2:(nk-1)], degree = 1, intercept = TRUE))
B2 <- cbind(Intercept = 1, bs(cdata$year, knots = naughts[2:(nk-1)], degree = 2, intercept = TRUE))

In [ ]:
B

In [ ]:
matplot(B[, -1], type = "l", lty = 1, lwd = 2, main = '1st degree basis function', ylab = "", xlab = "")
abline(v = 400)
# savefig('basis_fun.jpg', width = 12)

In [ ]:
matplot(B2[, -1], type = "l", lty = 1, lwd = 2, main = '2nd degree basis function', ylab = "", xlab = "")
abline(v = 400)
# savefig('basis_fun2.jpg', width = 12)

The werid thing about this is that it is just a linear model, which we can calculate in Stan. Here the basis matrix `B` is passed in as data, and we use a `transformed parameters` block to store `mu` (the equivalent of a `pm.Deterministic` in PyMC). Because the number of basis functions `K` is data, the same compiled model works for any spline degree or number of knots:

In [ ]:
# Spline model in Stan
temps_code <- "
data {
  int<lower=0> N;
  int<lower=0> K;      // number of basis functions
  matrix[N, K] B;      // basis function matrix
  vector[N] temp;
}
parameters {
  real b0;
  vector[K] w;
  real<lower=0> sigma;
}
transformed parameters {
  vector[N] mu = b0 + B*w;
}
model {
  // Priors
  b0 ~ normal(6, 10);
  w ~ normal(0, 1);
  sigma ~ exponential(1);

  // Likelihood
  temp ~ normal(mu, sigma);
}
"
temps <- cmdstan_model(write_stan_file(temps_code))

In [ ]:
# Sampling
trace_t <- temps$sample(data = list(N = nrow(cdata), K = ncol(B), B = B, temp = cdata$temp),
                        chains = 4, parallel_chains = 4, refresh = 0, show_exceptions = FALSE)

Looking back at our equation:

$$
\mu_i = \beta_0 + w_1 B_{i,1}+ w_2 B_{i,2}+...++ w_n B_{i,n}
$$

we 'simply' need to add these all together

In [ ]:
# Grab the posterior means
b0 <- mean(trace_t$draws("b0"))
w <- colMeans(trace_t$draws("w", format = "matrix"))

In [ ]:
w

In [ ]:
# Solve for mean spline value at each observation
spline_ <- b0 + B %*% w

In [ ]:
plot(cdata$year, cdata$temp, col = "dodgerblue", xlab = 'Year', ylab = 'March temp')
lines(cdata$year, spline_, lwd = 3)
abline(h = max(cdata$temp[cdata$year < 1800]), col = 'red')
# savefig('blossom_spline.jpg', width = 12)

Let's take a look at 2nd order - we can re-use the same compiled Stan model, just giving it the `B2` basis matrix:

In [ ]:
# Data for the 2nd degree spline
temps2_data <- list(N = nrow(cdata), K = ncol(B2), B = B2, temp = cdata$temp)

In [ ]:
# Sampling
trace_t2 <- temps$sample(data = temps2_data, chains = 4, parallel_chains = 4, refresh = 0, show_exceptions = FALSE)

In [ ]:
# Grab the posterior means
b0 <- mean(trace_t2$draws("b0"))
w <- colMeans(trace_t2$draws("w", format = "matrix"))

In [ ]:
# Solve for mean spline value at each observation
spline_ <- b0 + B2 %*% w

In [ ]:
plot(cdata$year, cdata$temp, col = "dodgerblue", xlab = 'Year', ylab = 'March temp')
lines(cdata$year, spline_, lwd = 3)
abline(h = max(cdata$temp[cdata$year < 1800]), col = 'red')
# savefig('blossom_spline2.jpg', width = 12)